In [3]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

# 1. Set random seed
seed = 22
tf.random.set_seed(seed)
np.random.seed(seed)

def debug_data_distribution(X_train, y_train, X_val, y_val, X_test, y_test):
    """Debug function to check data distribution and potential issues."""
    print("\n" + "="*60)
    print("DATA DISTRIBUTION ANALYSIS")
    print("="*60)

    # Class distribution
    print("Class distribution:")
    print(f"Train: {dict(zip(*np.unique(y_train, return_counts=True)))}")
    print(f"Val:   {dict(zip(*np.unique(y_val, return_counts=True)))}")
    print(f"Test:  {dict(zip(*np.unique(y_test, return_counts=True)))}")

    # Value ranges
    print(f"\nData value ranges:")
    print(f"Train: min={X_train.min():.4f}, max={X_train.max():.4f}, mean={X_train.mean():.4f}")
    print(f"Val:   min={X_val.min():.4f}, max={X_val.max():.4f}, mean={X_val.mean():.4f}")
    print(f"Test:  min={X_test.min():.4f}, max={X_test.max():.4f}, mean={X_test.mean():.4f}")

    # Shapes
    print(f"\nData shapes:")
    print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

    # Mean activations
    train_mean = X_train.mean(axis=(1,2,3))
    val_mean = X_val.mean(axis=(1,2,3))
    print(f"\nMean activation per sample:")
    print(f"Train mean: {train_mean.mean():.4f} ± {train_mean.std():.4f}")
    print(f"Val mean:   {val_mean.mean():.4f} ± {val_mean.std():.4f}")

    # Warnings
    if len(np.unique(y_val)) != len(np.unique(y_train)):
        print("⚠️ WARNING: Validation set doesn't contain all classes!")

    if X_val.std() == 0 or X_train.std() == 0:
        print("⚠️ WARNING: Zero std detected - check your data!")

    if abs(X_train.mean() - X_val.mean()) > 0.5:
        print("⚠️ WARNING: Large mean difference between train/val sets!")

    print("="*60)

def create_balanced_dataset(shape_input, file,
                           values_name, labels_name):
    """
    Load dataset and balance classes.
    balance_strategy = 'undersample' or 'oversample'
    """
    if len(shape_input) == 3:
        height, width, channels = shape_input
    elif len(shape_input) == 2:
        height, width = shape_input
        channels = 1
    else:
        raise ValueError("Shape Input must be (h, w) or (h, w, c).")

    # Load
    dataset = np.load(file)
    samples = dataset[values_name][:, :height, :width]
    classes = dataset[labels_name].astype(int)
    unique_classes = np.unique(classes)

    print(f"Original dataset: {samples.shape}, classes: {unique_classes}")
    print(f"Original class distribution: {dict(zip(*np.unique(classes, return_counts=True)))}")
    # Split
    X_temp, X_test, y_temp, y_test = train_test_split(
        samples, classes, test_size=0.2, stratify=classes, random_state=seed
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=seed
    )

    if channels == 1:
        X_train = np.expand_dims(X_train, -1)
        X_val = np.expand_dims(X_val, -1)
        X_test = np.expand_dims(X_test, -1)

    # Debug
    debug_data_distribution(X_train, y_train, X_val, y_val, X_test, y_test)

    return X_train, y_train, X_val, y_val, X_test, y_test, len(unique_classes)

shape_input = (40, 40, 1)
X_train, y_train, X_val, y_val, X_test, y_test, n_classes = create_balanced_dataset(
    shape_input, "dataset_kws_multi.npz", 'features', 'labels'
)
class_names=['no', 'up', 'down', 'unknown', 'yes']
X_test_flat=X_test.reshape(X_test.shape[0], -1).astype(np.float32)
print("Classes Number: ", len(class_names))
print("Names: ", class_names)

#teacher_dvec_model = tf.keras.models.load_model("./models/teacher_sv_cnn_model.h5")
model = tf.keras.models.load_model("100_kws_multi_final_deployable_student_model.h5")
model.summary()

Original dataset: (27583, 40, 40), classes: [0 1 2 3 4]
Original class distribution: {np.int64(0): np.int64(3929), np.int64(1): np.int64(3715), np.int64(2): np.int64(3908), np.int64(3): np.int64(11994), np.int64(4): np.int64(4037)}

DATA DISTRIBUTION ANALYSIS
Class distribution:
Train: {np.int64(0): np.int64(2357), np.int64(1): np.int64(2229), np.int64(2): np.int64(2345), np.int64(3): np.int64(7196), np.int64(4): np.int64(2422)}
Val:   {np.int64(0): np.int64(786), np.int64(1): np.int64(743), np.int64(2): np.int64(781), np.int64(3): np.int64(2399), np.int64(4): np.int64(808)}
Test:  {np.int64(0): np.int64(786), np.int64(1): np.int64(743), np.int64(2): np.int64(782), np.int64(3): np.int64(2399), np.int64(4): np.int64(807)}

Data value ranges:
Train: min=0.0000, max=1.0000, mean=0.2987
Val:   min=0.0000, max=1.0000, mean=0.2994
Test:  min=0.0000, max=1.0000, mean=0.2991

Data shapes:
Train: (16549, 40, 40, 1), Val: (5517, 40, 40, 1), Test: (5517, 40, 40, 1)

Mean activation per sample:
Tr

Model: "final_deployable_student_fusion"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1_fusion (Dense)          │ (None, 256)            │       409,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2_fusion (Dense)          │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3_fusion (Dense)          │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ class_output (Dense)            │ (None, 5)              │         1,285 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 542,725 (2.07 MB)

 Trainable params: 542,725 (2.07 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
def representative_data_gen():
  for i in range(100):
    yield[X_test_flat[i:i+1]]

print("Converting to INT8 TFLite Format...")
converter=tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations=[tf.lite.Optimize.DEFAULT]
converter.representative_dataset=representative_data_gen
converter.target_spec.supported_ops=[tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type=tf.int8
converter.inference_output_type=tf.int8

tflite_model_int8=converter.convert()
tflite_path="student_model_int8.tflite"
with open(tflite_path, 'wb') as f:
  f.write(tflite_model_int8)
print(f"Quantized model saved to {tflite_path}")

Converting to INT8 TFLite Format...
Saved artifact at '/tmp/tmpcnyw0ikd'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 1600), dtype=tf.float32, name='input')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  133527955615056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133527955615440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133527955616016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133527955616592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133527955616784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133527955616400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133527955615248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133527955614864: TensorSpec(shape=(), dtype=tf.resource, name=None)


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Quantized model saved to student_model_int8.tflite


In [8]:
def evaluate_tflite(tflite_path, x_data):
  interpreter=tf.lite.Interpreter(model_path=tflite_path)
  interpreter.allocate_tensors()
  input_details=interpreter.get_input_details()[0]
  output_details=interpreter.get_output_details()[0]
  is_int8=(input_details['dtype']==np.int8)
  preds=[]
  for i in range(len(x_data)):
    input_sample=x_data[i:i+1]
    if is_int8:
      scale, zero_point=input_details["quantization"]
      input_sample=np.int8(input_sample/scale+zero_point)
    interpreter.set_tensor(input_details["index"], input_sample)
    interpreter.invoke()
    output=interpreter.get_tensor(output_details["index"])
    preds.append(output.flatten())
  return np.array(preds)

print("\nRunning inference to compute Accuracy Delta...")
keras_preds=model.predict(X_test_flat, verbose=0)
keras_acc=np.mean(np.argmax(keras_preds, axis=1)==y_test)
tflite_preds=evaluate_tflite(tflite_path, X_test_flat)
tflite_acc=np.mean(np.argmax(tflite_preds, axis=1)==y_test)

print("\n"+"="*30)
print("QUANTIZATION ASSESSMENT")
print("="*30)
print(f"FP32 Model Accuracy: {keras_acc:.4f}")
print(f"INT8 Model Accuracy: {tflite_acc:.4f}")
print(f"Accuracy Delta: {abs(keras_acc-tflite_acc):.4f}")
print("="*30)


Running inference to compute Accuracy Delta...

QUANTIZATION ASSESSMENT
FP32 Model Accuracy: 0.8970
INT8 Model Accuracy: 0.8958
Accuracy Delta: 0.0013
